In [2]:
# Código para ejecutar en Google Colab: comparación interactiva de KMeans, Soft KMeans y GMM.
# Copia y pega todo en una celda de Colab y ejecútala. Si Colab te pide reiniciar el kernel tras instalar paquetes, hazlo y vuelve a ejecutar la celda.

!pip install -q scikit-learn ipywidgets

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Checkbox
from IPython.display import display, HTML
from scipy.stats import multivariate_normal

# =========================
# Generación de datasets
# =========================
def make_dataset(kind='Blobs', n_samples=400, noise=0.05, random_state=0):
    if kind == 'Blobs':
        X, y = make_blobs(n_samples=n_samples, centers=4, cluster_std=0.7, random_state=random_state)
    elif kind == 'Moons':
        X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
    elif kind == 'Circles':
        X, y = make_circles(n_samples=n_samples, noise=noise, factor=0.5, random_state=random_state)
    elif kind == 'Anisotropic':
        X, y = make_blobs(n_samples=n_samples, centers=4, random_state=random_state)
        T = np.array([[0.6, -0.6], [-0.4, 0.8]])
        X = X @ T
    return X, y

# =========================
# Soft K-Means
# =========================
def soft_kmeans(X, n_clusters=3, beta=1.0, max_iter=200, tol=1e-4, init_centers=None):
    n, d = X.shape
    rng = np.random.RandomState(0)

    if init_centers is None:
        idx = rng.choice(n, n_clusters, replace=False)
        mus = X[idx].astype(float)
    else:
        mus = init_centers.astype(float)

    prev_mus = mus.copy()

    for it in range(max_iter):
        d2 = np.sum((X[:, None, :] - mus[None, :, :])**2, axis=2)
        a = -beta * d2
        a -= a.max(axis=1, keepdims=True)
        R = np.exp(a)
        R /= R.sum(axis=1, keepdims=True)

        denom = R.sum(axis=0) + 1e-12
        mus = (R.T @ X) / denom[:, None]

        if np.linalg.norm(mus - prev_mus) < tol:
            break
        prev_mus = mus.copy()

    weights = R.sum(axis=0) / n
    return mus, R, weights, it + 1

# =========================
# Aproximación de densidad
# =========================
def approx_density(X, centers, R=None, grid=None):
    K = centers.shape[0]

    if R is None:
        sigma = np.median(np.linalg.norm(X - X.mean(0), axis=1))
        sigmas = np.full(K, sigma)
        weights = np.ones(K) / K
    else:
        denom = R.sum(axis=0) + 1e-12
        sigmas = []
        for k in range(K):
            diff = X - centers[k]
            var = (R[:,k] * np.sum(diff**2, axis=1)).sum() / (denom[k] * X.shape[1] + 1e-12)
            sigmas.append(np.sqrt(max(var, 1e-3)))
        sigmas = np.array(sigmas)
        weights = denom / denom.sum()

    if grid is None:
        return None

    XX = np.c_[grid[0].ravel(), grid[1].ravel()]
    dens = np.zeros(len(XX))

    for k in range(K):
        cov = np.eye(2) * (sigmas[k]**2 + 1e-6)
        mv = multivariate_normal(mean=centers[k], cov=cov, allow_singular=True)
        dens += weights[k] * mv.pdf(XX)

    return dens.reshape(grid[0].shape)

# =========================
# Visualización principal
# =========================
def plot_comparison(dataset='Blobs', n_samples=400, n_clusters=4, random_state=0,
                    beta=1.0, gmm_cov_type='full', show_density=True, grid_res=150):

    X, _ = make_dataset(dataset, n_samples=n_samples, random_state=random_state)

    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state).fit(X)
    labels_k = kmeans.labels_
    centers_k = kmeans.cluster_centers_

    centers_sk, R, w, iters = soft_kmeans(X, n_clusters=n_clusters, beta=beta, init_centers=centers_k)
    labels_sk = np.argmax(R, axis=1)

    gmm = GaussianMixture(n_components=n_clusters, covariance_type=gmm_cov_type,
                          random_state=random_state, n_init=3).fit(X)
    labels_g = gmm.predict(X)

    x_min, x_max = X[:,0].min()-0.8, X[:,0].max()+0.8
    y_min, y_max = X[:,1].min()-0.8, X[:,1].max()+0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_res),
                         np.linspace(y_min, y_max, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]

    d2_k = np.sum((grid[:,None,:] - centers_k[None,:,:])**2, axis=2)
    grid_labels_k = np.argmin(d2_k, axis=1).reshape(xx.shape)

    d2_sk = np.sum((grid[:,None,:] - centers_sk[None,:,:])**2, axis=2)
    Rg = np.exp(-beta * d2_sk)
    Rg /= Rg.sum(axis=1, keepdims=True)
    grid_labels_sk = np.argmax(Rg, axis=1).reshape(xx.shape)

    grid_labels_g = gmm.predict(grid).reshape(xx.shape)
    dens_gmm = np.exp(gmm.score_samples(grid)).reshape(xx.shape)
    dens_sk = approx_density(X, centers_sk, R=R, grid=(xx, yy))
    dens_k = approx_density(X, centers_k, R=None, grid=(xx, yy))

    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    titles = ["K-Means (Geométrico)", "Soft K-Means (Probabilidades)", "GMM (Probabilístico)"]

    for ax, title in zip(axes, titles):
        ax.set_title(title)

    axes[0].contourf(xx, yy, grid_labels_k, alpha=0.15)
    if show_density: axes[0].contour(xx, yy, dens_k, levels=8)
    axes[0].scatter(X[:,0], X[:,1], c=labels_k, s=20)
    axes[0].scatter(centers_k[:,0], centers_k[:,1], s=120, marker='X')

    axes[1].contourf(xx, yy, grid_labels_sk, alpha=0.15)
    if show_density: axes[1].contour(xx, yy, dens_sk, levels=8)
    axes[1].scatter(X[:,0], X[:,1], c=labels_sk, s=20)
    axes[1].scatter(centers_sk[:,0], centers_sk[:,1], s=120, marker='X')

    axes[2].contourf(xx, yy, grid_labels_g, alpha=0.15)
    if show_density: axes[2].contour(xx, yy, dens_gmm, levels=8)
    axes[2].scatter(X[:,0], X[:,1], c=labels_g, s=20)
    axes[2].scatter(gmm.means_[:,0], gmm.means_[:,1], s=120, marker='X')

    plt.tight_layout()
    plt.show()
    display(HTML(f"<b>Iteraciones Soft K-Means:</b> {iters} | <b>Beta:</b> {beta} | <b>GMM cov:</b> {gmm_cov_type}"))

# =========================
# Interfaz interactiva
# =========================
interact(plot_comparison,
         dataset=Dropdown(options=['Blobs','Moons','Circles','Anisotropic'], value='Blobs'),
         n_samples=IntSlider(min=100, max=1000, step=50, value=400),
         n_clusters=IntSlider(min=2, max=8, step=1, value=4),
         random_state=IntSlider(min=0, max=9999, step=1, value=0),
         beta=FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0),
         gmm_cov_type=Dropdown(options=['full','tied','diag','spherical'], value='full'),
         show_density=Checkbox(value=True));


interactive(children=(Dropdown(description='dataset', options=('Blobs', 'Moons', 'Circles', 'Anisotropic'), va…